# Chapitre 14 - Exercices 14.7 à 14.12

Composantes principales, problèmes de Procrustes et formes moyennes, positionnement
multidimensionnel classique, ACP parcimonieuse.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch14_non_supervise as ns
set_style()

## Un lemme utile : maximiser une trace sous contrainte d'orthonormalité

Plusieurs exercices utilisent le résultat suivant (inégalité de Ky Fan).

**Lemme.** Soit $\mathbf{S}$ une matrice $n \times n$ symétrique de valeurs propres $\lambda_1 \ge
\dots \ge \lambda_n$ et de vecteurs propres orthonormés $e_1, \dots, e_n$. Pour toute matrice
$\mathbf{M}$ ($n \times q$) telle que $\mathbf{M}^T\mathbf{M} = \mathbf{I}_q$,

$$\operatorname{tr}(\mathbf{M}^T\mathbf{S}\mathbf{M}) \le \lambda_1 + \dots + \lambda_q ,$$

avec égalité pour $\mathbf{M} = (e_1, \dots, e_q)$.

*Preuve.* On écrit $\mathbf{S} = \sum_j \lambda_j e_j e_j^T$, d'où
$\operatorname{tr}(\mathbf{M}^T\mathbf{S}\mathbf{M}) = \sum_j \lambda_j \operatorname{tr}(\mathbf{M}^Te_je_j^T\mathbf{M})
= \sum_j \lambda_j w_j$ avec $w_j = \|\mathbf{M}^Te_j\|^2$.

* $w_j \ge 0$, et $w_j = e_j^T\mathbf{M}\mathbf{M}^Te_j \le \|e_j\|^2 = 1$ car $\mathbf{M}\mathbf{M}^T$ est
  un projecteur orthogonal (il est symétrique et $(\mathbf{M}\mathbf{M}^T)^2 = \mathbf{M}\mathbf{M}^T$).
* $\sum_j w_j = \operatorname{tr}(\mathbf{M}^T(\sum_j e_je_j^T)\mathbf{M}) = \operatorname{tr}(\mathbf{M}^T\mathbf{M}) = q$.

Maximiser $\sum_j \lambda_j w_j$ sous $0 \le w_j \le 1$, $\sum_j w_j = q$ revient à mettre tout le
poids sur les $q$ plus grands $\lambda_j$ : le maximum vaut $\lambda_1 + \dots + \lambda_q$. Il est
atteint par $\mathbf{M} = (e_1, \dots, e_q)$, pour qui $w_1 = \dots = w_q = 1$. $\square$

In [2]:
rng = np.random.RandomState(0)
A = rng.normal(size=(6, 6))
S = A + A.T
lam = np.sort(np.linalg.eigvalsh(S))[::-1]
essais = [np.trace(Q.T @ S @ Q) for Q in (np.linalg.qr(rng.normal(size=(6, 2)))[0] for _ in range(2000))]
print("max sur 2000 M aléatoires : %.4f  <=  lambda_1 + lambda_2 = %.4f" % (max(essais), lam[:2].sum()))

max sur 2000 M aléatoires : 11.7060  <=  lambda_1 + lambda_2 = 13.1184


## Exercice 14.7 : Composantes principales : estimation de $\mu$ et des $\lambda_i$

**Énoncé.** Établir (14.51) et (14.52) de la section 14.5.1 : dans le problème
$\min_{\mu, \{\lambda_i\}, \mathbf{V}_q} \sum_i \|x_i - \mu - \mathbf{V}_q\lambda_i\|^2$ (14.50), où
$\mathbf{V}_q$ ($p \times q$) a des colonnes orthonormées, on peut prendre $\hat\mu = \bar x$ et
$\hat\lambda_i = \mathbf{V}_q^T(x_i - \bar x)$. Montrer que $\hat\mu$ n'est pas unique et
caractériser la famille des solutions équivalentes.

### Étape 1 : $\lambda_i$ à $\mu$ et $\mathbf{V}_q$ fixés

Les termes de la somme sont indépendants : pour chaque $i$, on minimise $\|y_i - \mathbf{V}_q\lambda_i\|^2$
avec $y_i = x_i - \mu$. C'est une régression par moindres carrés de $y_i$ sur les colonnes de
$\mathbf{V}_q$ ; les équations normales donnent $\mathbf{V}_q^T\mathbf{V}_q\lambda_i = \mathbf{V}_q^Ty_i$, et
comme $\mathbf{V}_q^T\mathbf{V}_q = \mathbf{I}_q$ :

$$\hat\lambda_i = \mathbf{V}_q^T(x_i - \mu).$$

Le résidu est $(\mathbf{I} - \mathbf{P})(x_i - \mu)$ avec $\mathbf{P} = \mathbf{V}_q\mathbf{V}_q^T$, projecteur
orthogonal sur le sous-espace engendré par les colonnes de $\mathbf{V}_q$.

### Étape 2 : $\mu$ à $\mathbf{V}_q$ fixé

Il reste à minimiser $F(\mu) = \sum_i \|(\mathbf{I} - \mathbf{P})(x_i - \mu)\|^2$. Notons $\mathbf{Q} =
\mathbf{I} - \mathbf{P}$, projecteur ($\mathbf{Q}^T = \mathbf{Q}$, $\mathbf{Q}^2 = \mathbf{Q}$). Le gradient
est

$$\nabla F(\mu) = -2\sum_i \mathbf{Q}^T\mathbf{Q}(x_i - \mu) = -2N\,\mathbf{Q}(\bar x - \mu).$$

$F$ est convexe (somme de carrés de fonctions affines) : ses minimiseurs sont les $\mu$ qui
annulent le gradient, c'est-à-dire $\mathbf{Q}(\mu - \bar x) = 0$, soit $\mu - \bar x \in
\operatorname{Im}(\mathbf{V}_q)$.

### Conclusion et famille des solutions

Les solutions sont exactement

$$\hat\mu = \bar x + \mathbf{V}_q c, \quad c \in \mathbb{R}^q \text{ quelconque}, \qquad
\hat\lambda_i = \mathbf{V}_q^T(x_i - \hat\mu) = \mathbf{V}_q^T(x_i - \bar x) - c .$$

Le choix $c = 0$ donne (14.51)-(14.52). Géométriquement, $\mu$ n'est qu'un point **quelconque**
du sous-espace affine $\bar x + \operatorname{Im}(\mathbf{V}_q)$ ; changer de point revient à
translater l'origine des coordonnées $\lambda_i$ le long du sous-espace, sans changer les
approximations $\hat\mu + \mathbf{V}_q\hat\lambda_i = \bar x + \mathbf{P}(x_i - \bar x)$.

> **Remarque.** $\mathbf{V}_q$ n'est pas unique non plus : pour toute matrice orthogonale $\mathbf{R}$
> ($q \times q$), $\mathbf{V}_q\mathbf{R}$ engendre le même sous-espace et donne le même projecteur ;
> seuls les $\lambda_i$ changent ($\mathbf{R}^T\lambda_i$). Avec $\hat\mu = \bar x$, il reste à
> minimiser $\sum_i\|(\mathbf{I} - \mathbf{P})\tilde x_i\|^2 = \|\tilde{\mathbf{X}}\|_F^2 -
> \operatorname{tr}(\mathbf{V}_q^T\tilde{\mathbf{X}}^T\tilde{\mathbf{X}}\mathbf{V}_q)$ (14.53), donc à maximiser
> la trace : par le lemme, les $q$ premiers vecteurs propres de $\tilde{\mathbf{X}}^T\tilde{\mathbf{X}}$,
> c'est-à-dire les $q$ premiers vecteurs singuliers à droite de $\tilde{\mathbf{X}}$.

### Vérification numérique

In [3]:
rng = np.random.RandomState(1)
X = rng.normal(size=(100, 4)) @ rng.normal(size=(4, 4))
a = ns.acp(X, 2)
V = a["V"]


def critere(mu, Lam):
    return ((X - mu - Lam @ V.T) ** 2).sum()


xbar = X.mean(0)
for c in (np.zeros(2), np.array([3.0, -1.0]), np.array([-10.0, 5.0])):
    mu = xbar + V @ c
    Lam = (X - mu) @ V
    print("c = %-12s critère = %.6f" % (c, critere(mu, Lam)))
mu_mauvais = xbar + rng.normal(size=4)
print("mu hors du sous-espace : critère = %.6f" % critere(mu_mauvais, (X - mu_mauvais) @ V))

c = [0. 0.]      critère = 227.221085
c = [ 3. -1.]    critère = 227.221085
c = [-10.   5.]  critère = 227.221085
mu hors du sous-espace : critère = 651.499673


Toutes les solutions $\bar x + \mathbf{V}_q c$ donnent le même critère (minimal) ; un $\mu$ hors de
cette famille fait strictement moins bien.